# Gemma 4 26B Single-Model Pipeline, Batched by Dimension

This version keeps one prompt per dimension, but runs **one full batched pass per dimension**
instead of four sequential calls per row. That is much faster and better aligned with vLLM.


## 1) Imports

In [2]:
import os
import gc
import json
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any

# Ensure vLLM child processes load the conda libstdc++ (fixes GLIBCXX_3.4.31 errors).
CONDA_LIBSTDCXX = Path("/home/cbenavent/envs/vllm312/lib/libstdc++.so.6")
if CONDA_LIBSTDCXX.exists():
    os.environ["LD_PRELOAD"] = str(CONDA_LIBSTDCXX)
    existing_ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    conda_lib_dir = str(CONDA_LIBSTDCXX.parent)
    if conda_lib_dir not in existing_ld_library_path.split(":"):
        os.environ["LD_LIBRARY_PATH"] = f"{conda_lib_dir}:{existing_ld_library_path}" if existing_ld_library_path else conda_lib_dir

import numpy as np
import pandas as pd
from IPython.display import display
from vllm import LLM, SamplingParams


## 2) Runtime parameters

In [3]:
MODEL_SPEC = {
    "label": "Gemma 4 26B",
    "model_name": "google/gemma-4-26B-A4B",
    "chat_mode": "manual_gemma",
    "dtype": "bfloat16",
    "gpu_memory_utilization": 0.82,
    "max_model_len": 4096,
    "max_new_tokens": 180,
    "max_num_seqs": 24,
    "disable_custom_all_reduce": True,
    "cuda_visible_devices": "0,1,2,3",
    "tensor_parallel_size": 4,
}

CUDA_VISIBLE_DEVICES = MODEL_SPEC["cuda_visible_devices"]
TENSOR_PARALLEL_SIZE = int(MODEL_SPEC["tensor_parallel_size"])

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("/home/cbenavent/test/Arcom_rares")

DATASET_PATH = PROJECT_ROOT / "data" / "annotation_working_master_human_2100_seed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "communication_function_outputs" / "per_model_dimension_runs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SMOKE_TEST_ROWS = 20
FULL_RUN_LIMIT = None

SCORE_MIN = 1.00
SCORE_MAX = 5.00
ROUND_DIGITS = 2
CHUNK_SIZE = 500
CONVERGENCE_DRIFT_THRESHOLD = 0.08

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
os.environ["OMP_NUM_THREADS"] = "8"

print("MODEL:", MODEL_SPEC["label"])
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TENSOR_PARALLEL_SIZE:", TENSOR_PARALLEL_SIZE)
print("MAX_NUM_SEQS:", MODEL_SPEC["max_num_seqs"])
print("LD_PRELOAD:", os.environ.get("LD_PRELOAD", ""))
print("VLLM_WORKER_MULTIPROC_METHOD:", os.environ.get("VLLM_WORKER_MULTIPROC_METHOD", ""))
print("OMP_NUM_THREADS:", os.environ.get("OMP_NUM_THREADS", ""))


MODEL: Gemma 4 26B
DATASET_PATH: /home/cbenavent/test/Arcom_rares/data/annotation_working_master_human_2100_seed.csv
OUTPUT_ROOT: /home/cbenavent/test/Arcom_rares/communication_function_outputs/per_model_dimension_runs
CUDA_VISIBLE_DEVICES: 0,1,2,3
TENSOR_PARALLEL_SIZE: 4
MAX_NUM_SEQS: 24
LD_PRELOAD: /home/cbenavent/envs/vllm312/lib/libstdc++.so.6
VLLM_WORKER_MULTIPROC_METHOD: spawn
OMP_NUM_THREADS: 8


## 3) Optional Hugging Face login

In [ ]:
os.environ["HF_TOKEN"] = "put your HF token here"



In [5]:
import os
from getpass import getpass
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN", "").strip()
if not hf_token:
    hf_token = getpass("Enter your Hugging Face token (input hidden): ").strip()

if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=False)
        print("HF login OK")
    except Exception as exc:
        print(f"HF login failed: {exc}")
else:
    print("No token provided.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK


## 4) Data and columns

In [6]:
df = pd.read_csv(DATASET_PATH)

SELECTED_COLUMNS = ["Script", "Titre", "Visuel", "Signature", "MotsClés", "Thème"]
ROW_ID_COL = "row_id"
SCORE_COLS = ["informativeness", "expressiveness", "phatic", "creativeness_poeticness"]
VALID_DOMINANT_DIMENSIONS = {
    "informativeness",
    "expressiveness",
    "phatic",
    "creativeness_poeticness",
    "mixed",
}

print("Columns currently used for model input:")
for col in SELECTED_COLUMNS:
    print("-", col)


Columns currently used for model input:
- Script
- Titre
- Visuel
- Signature
- MotsClés
- Thème


## 5) Prompt templates

In [7]:
COMMON_SYSTEM_PROMPT = """You are an expert annotation assistant for French automotive TV advertising transcription.

You must annotate carefully, conservatively, and consistently.
Your goal is to produce the most defensible annotation possible from the evidence in the ad.

You will receive ad text that may concatenate:
- Script
- Titre
- Visuel
- Signature
- MotsClés
- Thème

These elements may contain both literal information and symbolic or rhetorical cues.
You must judge the ad as a whole.

You are scoring only one dimension at a time.
Do not score the other dimensions.
Do not mention the other dimensions.
Do not let the presence of another rhetorical style mechanically raise this dimension unless it directly supports the dimension being scored.

Score the requested dimension from 1.00 to 5.00.
- 1.00 = almost absent
- 2.00 = weak
- 3.00 = moderate
- 4.00 = strong
- 5.00 = very strong

Intermediate decimal values are allowed.
Always return values rounded to two decimals.

Return strict JSON only.
Use exactly this schema:
{
  "score": 1.00,
  "confidence": 0.00,
  "reason": "short explanation or null"
}
"""

DIMENSION_INSTRUCTIONS = {
    "informativeness": """Dimension: INFORMATIVENESS

Definition:
How much the ad provides factual, concrete, product-related, offer-related, or technically useful information.

This includes:
- vehicle specifications
- features and equipment
- engine or powertrain information
- electric or hybrid technology
- charging, range, battery, consumption
- safety systems
- comfort or space features when presented concretely
- maintenance, guarantee, reliability claims when concrete
- price
- discounts
- financing, leasing, monthly payments
- trade-in conditions
- bonus or subsidy information
- model names, versions, product details
- explicit comparative or functional claims

High informativeness means the ad gives the viewer usable product or offer information.
Important rule: price, financing, technical details, and offer conditions are informative even if the ad is also emotional.
Do not raise this score just because the ad is stylish, poetic, playful, or relational unless those elements also add concrete information.
""",
    "expressiveness": """Dimension: EXPRESSIVENESS

Definition:
How much the ad relies on emotion, desire, identity, aspiration, style, symbolic value, atmosphere, prestige, seduction, or aesthetic projection.

This includes:
- emotional appeal
- beauty, elegance, sensuality
- pleasure, passion, freedom, adventure
- self-image and identity
- desire and dream
- luxury and prestige
- symbolic staging
- strong aestheticization
- dramatic mood
- poetic or evocative language
- brand mythology
- visual spectacle used to create attraction rather than explain the product

High expressiveness means the ad primarily tries to make the car or brand desirable, meaningful, aspirational, stylish, moving, or emotionally charged.
Do not raise this score only because the ad is informative, conversational, or joke-based unless those features directly intensify emotional or symbolic appeal.
""",
    "phatic": """Dimension: PHATIC

Definition:
How much the ad creates, maintains, or foregrounds social contact, relational connection, conversational closeness, complicity, or audience bonding.

This includes:
- direct address to the viewer
- rhetorical interaction
- social or conversational tone
- familiar, intimate, complicit language
- greetings, invitations, banter, playful contact
- communication whose main role is to establish or maintain connection rather than inform or aesthetically seduce
- emphasis on interpersonal exchange, contact, or togetherness
- “we are talking to you” energy
- casual rapport-building discourse

High phatic means the ad is strongly oriented toward establishing or maintaining a relationship with the audience or between people.
Important rule: humor alone does not automatically mean phatic.
Do not raise this score only because the ad is emotional, poetic, or creative unless the main function is social connection or audience bonding.
""",
    "creativeness_poeticness": """Dimension: CREATIVENESS_POETICNESS

Definition:
How much the ad relies on imaginative, poetic, playful, witty, surprising, metaphorical, or joke-based rhetorical construction.

This includes:
- poetic or highly evocative formulation
- metaphor, analogy, symbolism, or imaginative framing
- wordplay or verbal playfulness
- joke-based or humorous construction when it is part of the creative style
- surprising narrative setup
- unusual or inventive rhetorical staging
- creative twists, irony, absurdity, or playful exaggeration
- strong emphasis on originality of expression rather than only information or emotion

High creativeness_poeticness means the ad strongly stands out through inventive, poetic, witty, or playfully constructed communication.
Important rule: humor or jokes should increase this dimension when they function mainly as creative rhetorical style.
Do not raise this score only because the ad is emotional, informative, or socially engaging unless the expression itself is notably inventive, poetic, or playfully constructed.
""",
}


## 6) Shared helpers

In [8]:
def slugify_model_name(value: str) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def clean_text(value: Any) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_model_input_text(row: pd.Series, selected_columns=SELECTED_COLUMNS) -> str:
    parts = []
    for col in selected_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                value = clean_text(value)
                if value and value.lower() not in {"nan", "none"}:
                    parts.append(f"{col}: {value}")
    return "\n".join(parts)


def build_input_text(row: pd.Series) -> str:
    return build_model_input_text(row, SELECTED_COLUMNS)


GEMMA_CHAT_TEMPLATE = "<bos><start_of_turn>user\n{content}<end_of_turn>\n<start_of_turn>model\n"


def build_dimension_prompt_content(dimension: str, ad_text: str) -> str:
    clean_ad_text = clean_text(ad_text)
    return "\n\n".join([
        COMMON_SYSTEM_PROMPT.strip(),
        DIMENSION_INSTRUCTIONS[dimension].strip(),
        "AD TO SCORE",
        clean_ad_text,
        "Return only strict JSON.",
    ])


def render_prompt_for_model(content: str, llm=None, chat_mode: str = "hf_auto") -> str:
    if chat_mode == "manual_gemma":
        return GEMMA_CHAT_TEMPLATE.format(content=content)
    if chat_mode == "plain" or llm is None:
        return content
    try:
        tokenizer = llm.get_tokenizer()
        messages = [{"role": "user", "content": content}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception as exc:
        print(f"[WARN] apply_chat_template failed, falling back to plain prompt: {exc}")
        return content


def build_prompt_for_dimension(dimension: str, input_text: str, llm=None, chat_mode: str = "hf_auto") -> str:
    content = build_dimension_prompt_content(dimension, input_text)
    return render_prompt_for_model(content, llm=llm, chat_mode=chat_mode)


def clamp_score(value: Any, default: float = 1.0) -> float:
    try:
        text = str(value).strip().replace(",", ".")
        match = re.search(r"-?\d+(?:\.\d+)?", text)
        parsed = float(match.group(0)) if match else float(default)
        score = round(parsed, ROUND_DIGITS)
    except Exception:
        score = float(default)
    score = max(SCORE_MIN, min(SCORE_MAX, score))
    return round(score, ROUND_DIGITS)


def extract_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip())
                if isinstance(payload, dict):
                    return payload
            except json.JSONDecodeError:
                continue
    raise ValueError("Could not extract JSON from model output.")


def parse_dimension_prediction(raw_text: str) -> tuple[dict, bool, str]:
    try:
        payload = extract_json_object(raw_text)
        score = clamp_score(payload.get("score", 1.0))
        try:
            confidence = round(float(payload.get("confidence", 0.5)), ROUND_DIGITS)
        except Exception:
            confidence = 0.5
        confidence = max(0.0, min(1.0, confidence))
        reason = payload.get("reason", None)
        if reason is None or (isinstance(reason, float) and pd.isna(reason)):
            reason = None
        else:
            reason = clean_text(reason) or None
        return {"score": score, "confidence": confidence, "reason": reason}, True, ""
    except Exception as exc:
        return {"score": None, "confidence": 0.0, "reason": None}, False, str(exc)


def combine_dimension_predictions(preds: dict[str, dict]) -> dict:
    scores = {dim: preds[dim]["score"] for dim in SCORE_COLS}
    valid_scores = {k: v for k, v in scores.items() if v is not None}
    if valid_scores:
        ranked = sorted(valid_scores.items(), key=lambda x: x[1], reverse=True)
        top_score = ranked[0][1]
        top_labels = [k for k, v in ranked if v == top_score]
        dominant_dimension = "mixed" if len(top_labels) > 1 else top_labels[0]
        dominant_dimension_score = round(top_score, ROUND_DIGITS)
    else:
        dominant_dimension = ""
        dominant_dimension_score = None
    confidence_values = [preds[dim]["confidence"] for dim in SCORE_COLS if preds[dim]["score"] is not None]
    confidence = round(float(np.mean(confidence_values)), ROUND_DIGITS) if confidence_values else 0.0
    reason_parts = [f"{dim}: {preds[dim]['reason']}" for dim in SCORE_COLS if preds[dim]["reason"]]
    reason = " | ".join(reason_parts[:2]) if reason_parts else None
    return {
        **scores,
        "dominant_dimension": dominant_dimension,
        "dominant_dimension_score": dominant_dimension_score,
        "confidence": confidence,
        "reason": reason,
    }


def default_prediction(reason: str, parse_ok: bool = False) -> dict:
    return {
        "prediction": {
            "informativeness": None,
            "expressiveness": None,
            "phatic": None,
            "creativeness_poeticness": None,
            "dominant_dimension": "",
            "dominant_dimension_score": None,
            "confidence": 0.0,
            "reason": reason,
        },
        "dimension_outputs": {dim: {"score": None, "confidence": 0.0, "reason": None, "raw_output": ""} for dim in SCORE_COLS},
        "dimension_parse": {dim: False for dim in SCORE_COLS},
        "parse_ok": parse_ok,
        "parse_error": "" if parse_ok else reason,
    }


## 7) Build input text

In [9]:
missing_columns = [col for col in [ROW_ID_COL] + SELECTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

sample_df = df.loc[:, [ROW_ID_COL] + SELECTED_COLUMNS].copy()
sample_df["model_input_text"] = sample_df.apply(build_input_text, axis=1)
sample_df = sample_df[sample_df["model_input_text"].str.strip() != ""].copy()

if FULL_RUN_LIMIT is not None:
    sample_df = sample_df.head(FULL_RUN_LIMIT).copy()

print("Rows available for scoring:", len(sample_df))
display(sample_df[[ROW_ID_COL] + SELECTED_COLUMNS + ["model_input_text"]].head(3))
print("\nExample model input:\n")
print(sample_df.iloc[0]["model_input_text"])


Rows available for scoring: 8938


,row_id,Script,Titre,Visuel,Signature,MotsClés,Thème,model_input_text
0,0,"L'homme (1) : "" Wou !\r\n (1) ... On va où Mon...",EN VOITURE,"En caméra cachée, un homme invite des personne...",NaN,CAMERA CACHEE,"INTERROGATION, HUMOUR AUTRE, NOCTURNE, TWITTER","Script: L'homme (1) : "" Wou ! (1) ... On va où..."
1,1,"Voix homme : "" 85 000 conducteurs vivent la se...",LA SENSATION 100% ELECTRIQUE,Un homme recharge sa NISSAN LEAF et traverse u...,NISSAN. Innovation that excites. Innover autre...,BALLON,"INNOVATION, UNIVERS URBAIN, ANGLAIS, EFFET, SI...","Script: Voix homme : "" 85 000 conducteurs vive..."
2,2,"La femme (1) : "" C'est votre voiture ?\r\n L'h...",LA MÉTÉORITE,Des journalistes interrogent un homme qui vien...,"CITROEN, creative technologie.","CHANCE , POMPIER , CAMERA , INTERVIEW , METEOR...","ORIGINE, GARANTIE, ESPACE, JOURNALISTIQUE, DIS...","Script: La femme (1) : "" C'est votre voiture ?..."



Example model input:

Script: L'homme (1) : " Wou ! (1) ... On va où Monsieur ? L'homme (2) : " Dans le 15ème ! " (1) ... Ça m'arrange pas des masses ! Je veux bien aller à bon port, mais vous me dites bon port ... La femme (3) : " Mais bon port c'est une expression ! (1) ... J'ai vraiment mieux à vous proposer. (3) ... C'est une blague ! " (1) ... Yes or no ? L'homme (4) : " No ! " L'homme (5) : " Non, non je rigole pas parce qu'après c'est long ! " (1) ... Vous aimez l'aventure ? La femme (6) : " L'aventure ? " La femme (7) : " On est tombé sur un artiste ! " La femme (8) : " C'est pas là le chemin, non ! " (1) ... Je voulais vous proposer un truc c'est dommage ! (1) ... Et vous, vous allez oser ? "
Titre: EN VOITURE
Visuel: En caméra cachée, un homme invite des personnes à prendre place à l'arrière de sa voiture, une RENAUT CAPTUR, et fait comme si il était taxi.
MotsClés: CAMERA CACHEE
Thème: INTERROGATION, HUMOUR AUTRE, NOCTURNE, TWITTER


## 8) Sanity checks

In [10]:
assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
assert len(sample_df) > 0, "No rows left after input-text construction."
assert sample_df[ROW_ID_COL].notna().all(), "Some row_id values are missing."
assert sample_df["model_input_text"].str.len().gt(0).all(), "Some model_input_text values are empty."
assert sample_df[SELECTED_COLUMNS].notna().any(axis=1).all(), "Some kept rows have no non-null values in the selected input columns."

disk = shutil.disk_usage(PROJECT_ROOT)
print(f"Disk free near PROJECT_ROOT: {disk.free / (1024**3):.1f} GB")


Disk free near PROJECT_ROOT: 4228.8 GB


## 9) Smoke test rows

In [11]:
smoke_df = sample_df.head(SMOKE_TEST_ROWS).copy()
print("Smoke test rows:", len(smoke_df))
display(smoke_df[[ROW_ID_COL] + SELECTED_COLUMNS].head(5))
display(smoke_df[[ROW_ID_COL, "model_input_text"]].head(3))


Smoke test rows: 20


,row_id,Script,Titre,Visuel,Signature,MotsClés,Thème
0,0,"L'homme (1) : "" Wou !\r\n (1) ... On va où Mon...",EN VOITURE,"En caméra cachée, un homme invite des personne...",NaN,CAMERA CACHEE,"INTERROGATION, HUMOUR AUTRE, NOCTURNE, TWITTER"
1,1,"Voix homme : "" 85 000 conducteurs vivent la se...",LA SENSATION 100% ELECTRIQUE,Un homme recharge sa NISSAN LEAF et traverse u...,NISSAN. Innovation that excites. Innover autre...,BALLON,"INNOVATION, UNIVERS URBAIN, ANGLAIS, EFFET, SI..."
2,2,"La femme (1) : "" C'est votre voiture ?\r\n L'h...",LA MÉTÉORITE,Des journalistes interrogent un homme qui vien...,"CITROEN, creative technologie.","CHANCE , POMPIER , CAMERA , INTERVIEW , METEOR...","ORIGINE, GARANTIE, ESPACE, JOURNALISTIQUE, DIS..."
3,3,"La femme (1) : "" C'est votre voiture ?\r\n L'h...",LE MÉTÉORITE,Des journalistes interrogent un homme qui vien...,"CITROEN, creative technologie.","CHANCE , POMPIER , CAMERA , INTERVIEW , METEOR...","ORIGINE, GARANTIE, ESPACE, JOURNALISTIQUE, DIS..."
4,4,"La femme (1) : "" C'est votre voiture ?\r\n L'h...",LE MÉTÉORITE,Des journalistes interrogent un homme qui vien...,"CITROEN, creative technologie.","CHANCE , POMPIER , CAMERA , INTERVIEW , METEOR...","ESPACE, JOURNALISTIQUE, DISCOURS DECALE AUTRE,..."


,row_id,model_input_text
0,0,"Script: L'homme (1) : "" Wou ! (1) ... On va où..."
1,1,"Script: Voix homme : "" 85 000 conducteurs vive..."
2,2,"Script: La femme (1) : "" C'est votre voiture ?..."


## 10) Inference helpers

In [12]:
def inspect_gpu_memory():
    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,memory.used,memory.free",
            "--format=csv,noheader,nounits",
        ]
        completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
        lines = []
        for line in completed.stdout.strip().splitlines():
            idx, name, total, used, free = [part.strip() for part in line.split(",", 4)]
            lines.append({
                "gpu_index": idx,
                "gpu_name": name,
                "total_gb": round(float(total) / 1024.0, 2),
                "used_gb": round(float(used) / 1024.0, 2),
                "free_gb": round(float(free) / 1024.0, 2),
            })
        return pd.DataFrame(lines)
    except Exception as exc:
        print("nvidia-smi inspection failed:", exc)
        return pd.DataFrame()


def build_model_kwargs(spec: dict[str, Any]) -> dict[str, Any]:
    kwargs = {
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "trust_remote_code": True,
        "gpu_memory_utilization": float(spec["gpu_memory_utilization"]),
        "dtype": spec["dtype"],
        "max_model_len": int(spec["max_model_len"]),
        "disable_log_stats": True,
    }
    if spec.get("disable_custom_all_reduce", False):
        kwargs["disable_custom_all_reduce"] = True
    if spec.get("max_num_seqs") is not None:
        kwargs["max_num_seqs"] = int(spec["max_num_seqs"])
    return kwargs


def summarise_chunk_convergence(results_df: pd.DataFrame, chunk_size: int = CHUNK_SIZE) -> dict[str, Any]:
    ok_df = results_df[results_df["parse_ok"] == True].copy()
    if ok_df.empty:
        return {
            "chunk_count": 0,
            "convergence_proxy": "no",
            "convergence_ratio": 0.0,
            "last_chunk_mean_drift": None,
        }
    ok_df = ok_df.reset_index(drop=True)
    ok_df["chunk_id"] = ok_df.index // chunk_size
    chunk_means = ok_df.groupby("chunk_id")[SCORE_COLS].mean()
    if len(chunk_means) < 3:
        return {
            "chunk_count": int(len(chunk_means)),
            "convergence_proxy": "not_enough_chunks",
            "convergence_ratio": None,
            "last_chunk_mean_drift": None,
        }
    recent = chunk_means.tail(3)
    drift = float(recent.max().sub(recent.min()).mean())
    convergence_ratio = max(0.0, 1.0 - (drift / 1.0))
    converged = drift <= CONVERGENCE_DRIFT_THRESHOLD
    return {
        "chunk_count": int(len(chunk_means)),
        "convergence_proxy": "yes" if converged else "no",
        "convergence_ratio": round(convergence_ratio, 4),
        "last_chunk_mean_drift": round(drift, 4),
    }


def _extract_generated_text(output) -> tuple[str, str]:
    if output is None:
        return "", "missing generation output"
    generated_items = getattr(output, "outputs", None)
    if not generated_items:
        return "", "empty generation output"
    first_item = generated_items[0]
    raw_text = getattr(first_item, "text", "") or ""
    return raw_text, "" if raw_text else "empty generated text"


def run_dimension_batch(llm, run_df: pd.DataFrame, dimension: str, spec: dict[str, Any]) -> pd.DataFrame:
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=int(spec["max_new_tokens"]),
    )
    prompts = [
        build_prompt_for_dimension(dimension, text, llm=llm, chat_mode=spec["chat_mode"])
        for text in run_df["model_input_text"].tolist()
    ]
    try:
        outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    except TypeError:
        outputs = llm.generate(prompts, sampling_params)

    rows = []
    for input_row, output in zip(run_df.itertuples(index=False), outputs):
        raw_text, extraction_error = _extract_generated_text(output)
        if extraction_error:
            parsed = {"score": None, "confidence": 0.0, "reason": None}
            parse_ok = False
            parse_error = extraction_error
        else:
            parsed, parse_ok, parse_error = parse_dimension_prediction(raw_text)
        if not parse_ok and extraction_error and not parse_error:
            parse_error = extraction_error
        rows.append({
            "row_id": int(getattr(input_row, ROW_ID_COL)) if pd.notna(getattr(input_row, ROW_ID_COL)) else None,
            f"{dimension}__score": parsed["score"],
            f"{dimension}__confidence": parsed["confidence"],
            f"{dimension}__reason": parsed["reason"],
            f"{dimension}__raw_output": raw_text,
            f"{dimension}__parse_ok": parse_ok,
            f"{dimension}__parse_error": parse_error,
        })
    return pd.DataFrame(rows)


def merge_dimension_batches(run_df: pd.DataFrame, dimension_frames: list[pd.DataFrame]) -> pd.DataFrame:
    merged = run_df[[ROW_ID_COL, "model_input_text"]].copy()
    for frame in dimension_frames:
        merged = merged.merge(frame, on="row_id", how="left")

    merged["informativeness"] = merged["informativeness__score"]
    merged["expressiveness"] = merged["expressiveness__score"]
    merged["phatic"] = merged["phatic__score"]
    merged["creativeness_poeticness"] = merged["creativeness_poeticness__score"]

    def build_combined(row):
        preds = {
            dim: {
                "score": row.get(f"{dim}__score"),
                "confidence": row.get(f"{dim}__confidence"),
                "reason": row.get(f"{dim}__reason"),
            } for dim in SCORE_COLS
        }
        return combine_dimension_predictions(preds)

    combined = merged.apply(build_combined, axis=1)
    merged["dominant_dimension"] = combined.map(lambda x: x["dominant_dimension"])
    merged["dominant_dimension_score"] = combined.map(lambda x: x["dominant_dimension_score"])
    merged["confidence"] = combined.map(lambda x: x["confidence"])
    merged["reason"] = combined.map(lambda x: x["reason"])

    parse_cols = [f"{dim}__parse_ok" for dim in SCORE_COLS]
    error_cols = [f"{dim}__parse_error" for dim in SCORE_COLS]
    merged["parse_ok"] = merged[parse_cols].fillna(False).all(axis=1)
    merged["parse_error"] = merged[error_cols].apply(
        lambda row: " | ".join(
            f"{dim}: {err}" for dim, err in zip(SCORE_COLS, row.tolist()) if isinstance(err, str) and err.strip()
        ),
        axis=1,
    )
    return merged


def run_model_on_dataframe(spec: dict[str, Any], run_df: pd.DataFrame, run_name: str):
    llm = None
    model_kwargs = build_model_kwargs(spec)
    perf_start = time.perf_counter()
    gpu_before = inspect_gpu_memory()

    try:
        llm = LLM(model=spec["model_name"], **model_kwargs)
        gpu_after_load = inspect_gpu_memory()
        dimension_frames = []
        for dimension in SCORE_COLS:
            print(f"Running batched pass for dimension: {dimension}")
            dimension_frames.append(run_dimension_batch(llm, run_df, dimension, spec))

        merged = merge_dimension_batches(run_df, dimension_frames)
        merged["model_name"] = spec["model_name"]
        merged["model_label"] = spec["label"]
        merged["run_name"] = run_name

    except Exception as exc:
        error_text = f"LLM failed: {exc}"
        print("[ERROR]", error_text)
        gpu_after_load = inspect_gpu_memory()
        merged = run_df[[ROW_ID_COL, "model_input_text"]].copy()
        merged["model_name"] = spec["model_name"]
        merged["model_label"] = spec["label"]
        merged["run_name"] = run_name
        for dim in SCORE_COLS:
            merged[f"{dim}__score"] = None
            merged[f"{dim}__confidence"] = 0.0
            merged[f"{dim}__reason"] = None
            merged[f"{dim}__raw_output"] = ""
            merged[f"{dim}__parse_ok"] = False
            merged[f"{dim}__parse_error"] = error_text
        merged["informativeness"] = None
        merged["expressiveness"] = None
        merged["phatic"] = None
        merged["creativeness_poeticness"] = None
        merged["dominant_dimension"] = ""
        merged["dominant_dimension_score"] = None
        merged["confidence"] = 0.0
        merged["reason"] = error_text
        merged["parse_ok"] = False
        merged["parse_error"] = error_text

    finally:
        try:
            if llm is not None:
                del llm
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

    perf_end = time.perf_counter()
    gpu_after_cleanup = inspect_gpu_memory()
    perf_seconds = max(0.0, perf_end - perf_start)
    throughput = (len(merged) / perf_seconds) if perf_seconds > 0 else float("inf")
    parse_ok_count = int(merged["parse_ok"].fillna(False).sum())

    diagnostics = {
        "model_label": spec["label"],
        "model_name": spec["model_name"],
        "run_name": run_name,
        "rows_requested": len(run_df),
        "rows_returned": len(merged),
        "parse_ok_count": parse_ok_count,
        "parse_fail_count": len(merged) - parse_ok_count,
        "parse_ok_rate": round(parse_ok_count / len(merged), 4) if len(merged) else 0.0,
        "total_seconds": round(perf_seconds, 3),
        "rows_per_second": round(throughput, 3),
        "gpu_before": gpu_before.to_dict(orient="records"),
        "gpu_after_load": gpu_after_load.to_dict(orient="records"),
        "gpu_after_cleanup": gpu_after_cleanup.to_dict(orient="records"),
    }
    return merged, diagnostics


def save_model_outputs(results_df: pd.DataFrame, diagnostics: dict[str, Any]) -> dict[str, Path]:
    model_slug = slugify_model_name(diagnostics["model_name"])
    model_dir = OUTPUT_ROOT / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    jsonl_path = model_dir / f"{model_slug}__{diagnostics['run_name']}.jsonl"
    full_csv_path = model_dir / f"per_model_total_output_{model_slug}.csv"
    score_csv_path = model_dir / f"output_dimnesions_scores_{model_slug}.csv"
    diag_csv_path = model_dir / f"{model_slug}__{diagnostics['run_name']}_diagnostics.csv"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in results_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    results_df.to_csv(full_csv_path, index=False)
    score_cols = [
        "row_id",
        "informativeness",
        "expressiveness",
        "phatic",
        "creativeness_poeticness",
        "dominant_dimension",
        "dominant_dimension_score",
        "confidence",
        "reason",
        "model_name",
        "model_label",
    ]
    results_df.loc[:, score_cols].to_csv(score_csv_path, index=False)
    pd.DataFrame([diagnostics]).to_csv(diag_csv_path, index=False)

    return {
        "jsonl_path": jsonl_path,
        "full_csv_path": full_csv_path,
        "score_csv_path": score_csv_path,
        "diag_csv_path": diag_csv_path,
    }


## 11) Smoke test

In [13]:
smoke_results, smoke_diag = run_model_on_dataframe(MODEL_SPEC, smoke_df, run_name="smoke_test_20_rows")
smoke_conv = summarise_chunk_convergence(smoke_results)
smoke_diag.update(smoke_conv)
display(smoke_results.head(3))
display(pd.DataFrame([smoke_diag])[[
    "model_label",
    "rows_requested",
    "rows_returned",
    "parse_ok_rate",
    "rows_per_second",
    "convergence_proxy",
    "convergence_ratio",
    "last_chunk_mean_drift",
]])


INFO 06-18 18:02:16 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 4096, 'tensor_parallel_size': 4, 'gpu_memory_utilization': 0.82, 'max_num_seqs': 24, 'disable_log_stats': True, 'disable_custom_all_reduce': True, 'model': 'google/gemma-4-26B-A4B'}


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

INFO 06-18 18:02:18 [model.py:549] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-18 18:02:18 [model.py:1678] Using max model len 4096
INFO 06-18 18:02:18 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-18 18:02:18 [config.py:104] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-18 18:02:18 [vllm.py:790] Asynchronous scheduling is enabled.
INFO 06-18 18:02:21 [compilation.py:292] Enabled custom fusions: allreduce_rms


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


tokenizer_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

...

Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=1125620) INFO 06-18 18:06:07 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=1125620) INFO 06-18 18:06:07 [core.py:1233] Shutdown complete
(Worker_TP3 pid=1125668) INFO 06-18 18:06:07 [multiproc_executor.py:859] WorkerProc shutting down.
(Worker_TP0 pid=1125665) INFO 06-18 18:06:07 [multiproc_executor.py:764] Parent process exited, terminating worker queues
(Worker_TP0 pid=1125665) INFO 06-18 18:06:07 [multiproc_executor.py:859] WorkerProc shutting down.
(Worker_TP1 pid=1125666) INFO 06-18 18:06:07 [multiproc_executor.py:859] WorkerProc shutting down.
(Worker_TP2 pid=1125667) INFO 06-18 18:06:07 [multiproc_executor.py:859] WorkerProc shutting down.


,row_id,model_input_text,informativeness__score,informativeness__confidence,informativeness__reason,informativeness__raw_output,informativeness__parse_ok,informativeness__parse_error,expressiveness__score,expressiveness__confidence,...,creativeness_poeticness,dominant_dimension,dominant_dimension_score,confidence,reason,parse_ok,parse_error,model_name,model_label,run_name
0,0,"Script: L'homme (1) : "" Wou ! (1) ... On va où...",1.0,0.00,NaN,"{\n ""score"": 1.00,\n ""confidence"": 0.00,\n ...",True,,4.5,0.95,...,3.5,expressiveness,4.5,0.66,informativeness: nan | expressiveness: The ad ...,True,,google/gemma-4-26B-A4B,Gemma 4 26B,smoke_test_20_rows
1,1,"Script: Voix homme : "" 85 000 conducteurs vive...",4.0,0.95,The ad provides detailed information about the...,"{\n ""score"": 4.00,\n ""confidence"": 0.95,\n ...",True,,4.5,0.95,...,4.5,mixed,4.5,0.71,informativeness: The ad provides detailed info...,False,phatic: Could not extract JSON from model output.,google/gemma-4-26B-A4B,Gemma 4 26B,smoke_test_20_rows
2,2,"Script: La femme (1) : "" C'est votre voiture ?...",1.0,0.00,NaN,"{\n ""score"": 1.00,\n ""confidence"": 0.00,\n ...",True,,4.5,0.95,...,4.0,expressiveness,4.5,0.47,informativeness: nan | expressiveness: The ad ...,True,,google/gemma-4-26B-A4B,Gemma 4 26B,smoke_test_20_rows


,model_label,rows_requested,rows_returned,parse_ok_rate,rows_per_second,convergence_proxy,convergence_ratio,last_chunk_mean_drift
0,Gemma 4 26B,20,20,0.85,0.085,not_enough_chunks,None,None


## 12) Full run

In [ ]:
full_results, full_diag = run_model_on_dataframe(MODEL_SPEC, sample_df, run_name=f"full_{len(sample_df)}_rows")
full_conv = summarise_chunk_convergence(full_results)
full_diag.update(full_conv)
export_paths = save_model_outputs(full_results, full_diag)
full_diag.update({k: str(v) for k, v in export_paths.items()})

full_diag_df = pd.DataFrame([full_diag])
display(full_diag_df)
display(full_diag_df[[
    "model_label",
    "rows_requested",
    "rows_returned",
    "parse_ok_count",
    "parse_fail_count",
    "parse_ok_rate",
    "rows_per_second",
    "convergence_proxy",
    "convergence_ratio",
    "last_chunk_mean_drift",
]])
display(full_diag_df[["model_label", "score_csv_path", "full_csv_path", "diag_csv_path"]])


INFO 06-18 18:06:13 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 4096, 'tensor_parallel_size': 4, 'gpu_memory_utilization': 0.82, 'max_num_seqs': 24, 'disable_log_stats': True, 'disable_custom_all_reduce': True, 'model': 'google/gemma-4-26B-A4B'}
INFO 06-18 18:06:14 [model.py:549] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-18 18:06:14 [model.py:1678] Using max model len 4096
INFO 06-18 18:06:14 [config.py:104] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
(EngineCore pid=1128435) INFO 06-18 18:06:22 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='google/gemma-4-26B-A4B', speculative_config=None, tokenizer='google/gemma-4-26B-A4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096

(Worker pid=1128486) (Worker pid=1128485) (Worker pid=1128487) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=1128486) (Worker pid=1128487) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecate

(Worker pid=1128485) INFO 06-18 18:06:37 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=1128485) WARNING 06-18 18:06:39 [symm_mem.py:106] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(Worker pid=1128487) WARNING 06-18 18:06:39 [symm_mem.py:106] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(Worker pid=1128488) WARNING 06-18 18:06:39 [symm_mem.py:106] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(Worker pid=1128486) WARNING 06-18 18:06:39 [symm_mem.py:106] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(Worker pid=1128485) INFO 06-18 18:06:39 [parallel_state.py:1716] rank 0 in world size 4 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(Worker_TP0 pid=1128485) INFO 06-18 18:06:39 [gpu_model_runner.py:4735] Starting to load model google/gemma-4-26B-A4B...
(Worker_TP2 pid=1128487) INFO 06-18 18:06:40 [cuda.py:274] Us

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:03<00:03,  3.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:04<00:00,  1.94s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:04<00:00,  2.18s/it]
(Worker_TP0 pid=1128485) 


(Worker_TP0 pid=1128485) INFO 06-18 18:06:45 [default_loader.py:384] Loading weights took 4.42 seconds
(Worker_TP0 pid=1128485) INFO 06-18 18:06:46 [gpu_model_runner.py:4820] Model loading took 13.29 GiB memory and 5.982558 seconds
(Worker_TP0 pid=1128485) INFO 06-18 18:06:47 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 6 video items of the maximum feature size.
(Worker_TP0 pid=1128485) INFO 06-18 18:07:24 [backends.py:1051] Using cache directory: /home/cbenavent/.cache/vllm/torch_compile_cache/405bfbf7ec/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=1128485) INFO 06-18 18:07:24 [backends.py:1111] Dynamo bytecode transform time: 2.84 s
(Worker_TP0 pid=1128485) INFO 06-18 18:07:24 [flashinfer_all_reduce.py:109] Auto-selected flashinfer allreduce backend: trtllm
(Worker_TP0 pid=1128485) WARNING 06-18 18:07:24 [flashinfer_all_reduce.py:65] Failed to initialize FlashInfer All Reduce workspace: [SymmDeviceMemory] 

...

Rendering prompts:   0%|          | 0/8938 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8938 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(Worker_TP0 pid=1128485) WARNING 06-18 18:08:27 [multiproc_executor.py:871] WorkerProc was terminated
(Worker_TP1 pid=1128486) (Worker_TP2 pid=1128487) WARNING 06-18 18:08:27 [multiproc_executor.py:871] WorkerProc was terminated
WARNING 06-18 18:08:27 [multiproc_executor.py:871] WorkerProc was terminated
(Worker_TP3 pid=1128488) WARNING 06-18 18:08:27 [multiproc_executor.py:871] WorkerProc was terminated


KeyboardInterrupt: 

: 